# Cybersecurity Threat Intelligence ML Layer

This notebook implements the Step 9 machine learning layer for the cybersecurity threat intelligence project. It loads the enriched CVE dataset, engineers features, clusters vulnerabilities, and creates a transparent `ml_priority_score` for demonstration and prioritization.

The project pipeline is:

NVD API
?
Spark + Scala
?
CVE normalization
?
CWE enrichment
?
Aggregations
?
Python ML
?
Clustering + prioritization

The current development dataset contains only 20 CVEs. This is a test/development implementation and should not be treated as a production-ready security model.


## 1. Imports

Import the libraries used for data loading, preprocessing, clustering, and simple visualizations.


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 2. Load dataset

Load the enriched CVE parquet dataset and inspect the shape and schema.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    search_start = (start or Path.cwd()).resolve()
    for candidate in [search_start, *search_start.parents]:
        if (candidate / 'data' / 'processed' / 'cves_enriched.parquet').exists():
            return candidate
    return search_start

project_root = find_project_root()
input_path = project_root / 'data' / 'processed' / 'cves_enriched.parquet'
df = pd.read_parquet(input_path)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("Column names:")
print(df.columns.tolist())
print("\nFirst rows:")
display(df.head())
print("\nDataset info:")
df.info()


Rows: 20
Columns: 14
Column names:
['cve_id', 'published', 'last_modified', 'description', 'cvss_score', 'severity', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope', 'cwe_id', 'cwe_name', 'cwe_description']

First rows:


,cve_id,published,last_modified,description,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,scope,cwe_id,cwe_name,cwe_description
0,CVE-1999-1467,1989-10-25 22:30:00,2026-06-16 16:20:30.147,Vulnerability in rcp on SunOS 4.0.x allows rem...,10.0,HIGH,NETWORK,LOW,NaN,NaN,NaN,NVD-CWE-Other,NaN,NaN
1,CVE-1999-1554,1990-10-30 23:30:00,2026-06-16 16:20:42.350,/usr/sbin/Mail on SGI IRIX 3.3 and 3.3.1 does ...,2.1,LOW,LOCAL,LOW,NaN,NaN,NaN,NVD-CWE-Other,NaN,NaN
2,CVE-1999-1391,1990-10-02 22:30:00,2026-06-16 16:20:20.377,Vulnerability in NeXT 1.0a and 1.0 with public...,7.2,HIGH,LOCAL,LOW,NaN,NaN,NaN,NVD-CWE-Other,NaN,NaN
3,CVE-1999-1197,1990-12-19 23:30:00,2026-06-16 16:19:55.497,TIOCCONS in SunOS 4.1.1 does not properly chec...,7.2,HIGH,LOCAL,LOW,NaN,NaN,NaN,NVD-CWE-Other,NaN,NaN
4,CVE-1999-1258,1991-01-14 23:30:00,2026-06-16 16:20:03.313,rpc.pwdauthd in SunOS 4.1.1 and earlier does n...,5.0,MEDIUM,NETWORK,LOW,NaN,NaN,NaN,NVD-CWE-Other,NaN,NaN



Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   cve_id               20 non-null     str           
 1   published            20 non-null     datetime64[ns]
 2   last_modified        20 non-null     datetime64[ns]
 3   description          20 non-null     str           
 4   cvss_score           20 non-null     float64       
 5   severity             20 non-null     str           
 6   attack_vector        20 non-null     str           
 7   attack_complexity    20 non-null     str           
 8   privileges_required  1 non-null      str           
 9   user_interaction     1 non-null      str           
 10  scope                1 non-null      str           
 11  cwe_id               20 non-null     str           
 12  cwe_name             1 non-null      str           
 13  cwe_description      1 non-null  

## 3. Exploratory data inspection

Review the distribution of key vulnerability characteristics before running clustering.


In [3]:
plt.figure(figsize=(8, 5))
df['cvss_score'].plot(kind='hist', bins=12, edgecolor='black')
plt.title('CVSS Score Distribution')
plt.xlabel('CVSS Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
df['severity'].value_counts().plot(kind='bar', edgecolor='black')
plt.title('Severity Distribution')
plt.xlabel('Severity')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
df['attack_vector'].value_counts().plot(kind='bar', edgecolor='black')
plt.title('Attack Vector Distribution')
plt.xlabel('Attack Vector')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print('CVSS summary:')
print(df['cvss_score'].describe())
print('\nSeverity counts:')
print(df['severity'].value_counts(dropna=False))
print('\nAttack vector counts:')
print(df['attack_vector'].value_counts(dropna=False))
print('\nPrivileges required counts:')
print(df['privileges_required'].value_counts(dropna=False))
print('\nUser interaction counts:')
print(df['user_interaction'].value_counts(dropna=False))
print('\nScope counts:')
print(df['scope'].value_counts(dropna=False))
print('\nCWE availability:')
print(df['cwe_id'].value_counts(dropna=False).head())


CVSS summary:
count    20.000000
mean      6.975000
std       1.964655
min       2.100000
25%       6.650000
50%       7.200000
75%       7.500000
max      10.000000
Name: cvss_score, dtype: float64

Severity counts:
severity
HIGH      15
MEDIUM     4
LOW        1
Name: count, dtype: int64

Attack vector counts:
attack_vector
LOCAL      13
NETWORK     7
Name: count, dtype: int64

Privileges required counts:
privileges_required
NaN     19
NONE     1
Name: count, dtype: int64

User interaction counts:
user_interaction
NaN     19
NONE     1
Name: count, dtype: int64

Scope counts:
scope
NaN          19
UNCHANGED     1
Name: count, dtype: int64

CWE availability:
cwe_id
NVD-CWE-Other    19
CWE-269           1
Name: count, dtype: int64


C:\Users\Riddhima\AppData\Local\Temp\ipykernel_15008\3340114128.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Riddhima\AppData\Local\Temp\ipykernel_15008\3340114128.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Riddhima\AppData\Local\Temp\ipykernel_15008\3340114128.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Feature selection

For the first unsupervised clustering implementation, we use meaningful vulnerability characteristics but avoid identifier and leakage-style variables.

Numerical feature:
- `cvss_score`

Categorical features:
- `attack_vector`
- `attack_complexity`
- `privileges_required`
- `user_interaction`
- `scope`
- `cwe_id`

These are used because they describe technical vulnerability characteristics relevant to grouping similar CVEs.

These are not used as clustering features:
- `cve_id` because it is an identifier
- `description` because it is unstructured text and not included in this first implementation
- `severity` because it is derived from CVSS and would create leakage/redundancy
- raw timestamps because they are not a vulnerability characteristic for this clustering task


## 5. Missing value handling

We use the same scikit-learn preprocessing strategy as the Python implementation.

- Numerical: median imputation
- Categorical: most-frequent imputation and OneHotEncoder


In [4]:
FEATURE_COLUMNS = ['cvss_score', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope', 'cwe_id']
NUMERIC_FEATURES = ['cvss_score']
CATEGORICAL_FEATURES = ['attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope', 'cwe_id']

def build_preprocessor():
    numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]
    )
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_transformer, NUMERIC_FEATURES),
            ('categorical', categorical_transformer, CATEGORICAL_FEATURES),
        ]
    )

preprocessor = build_preprocessor()
print(preprocessor)


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['cvss_score']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['attack_vector', 'attack_complexity',
                                  'privileges_required', 'user_interaction',
                                  'scope', 'cwe_id'])])


## 6. Categorical encoding

Nominal categorical values such as `attack_vector` and `scope` should not be converted to arbitrary integers because those integers imply a false ordering. One-hot encoding preserves category meaning without imposing a ranking that does not exist.


## 7. Build feature matrix

Create the model dataset with the selected features, apply preprocessing, and inspect the transformed matrix.


In [5]:
model_df = df[FEATURE_COLUMNS].copy()
model_df = model_df.replace({'': np.nan, ' ': np.nan})
print(f"Original feature count: {model_df.shape[1]}")
feature_matrix = preprocessor.fit_transform(model_df)
print(f"Transformed feature count: {feature_matrix.shape[1]}")
print(f"Feature matrix shape: {feature_matrix.shape}")


Original feature count: 7
Transformed feature count: 9
Feature matrix shape: (20, 9)


## 8. K-Means clustering

This uses K-Means on the engineered feature matrix. The current development dataset only contains 20 CVEs, so a small cluster count is appropriate. The default is `k=3`, with `random_state=42` and `n_init=10`. The cluster IDs are arbitrary and not inherently low, medium, or high risk.


In [6]:
k_value = 3
n_clusters = min(k_value, len(model_df))
if len(model_df) < 3:
    n_clusters = max(1, len(model_df))

kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(feature_matrix)
results = df.copy()
results['cluster_id'] = cluster_labels
print(f"K-Means clusters created: {len(np.unique(cluster_labels))}")
print(results['cluster_id'].value_counts().sort_index())


K-Means clusters created: 3
cluster_id
0    12
1     5
2     3
Name: count, dtype: int64


## 9. Cluster analysis

Inspect the cluster size and CVSS statistics without assigning a risk label to the numeric cluster IDs.


In [7]:
cluster_summary_df = (
    results.groupby('cluster_id', as_index=False)
    .agg(
        cve_count=('cve_id', 'count'),
        average_cvss=('cvss_score', 'mean'),
        minimum_cvss=('cvss_score', 'min'),
        maximum_cvss=('cvss_score', 'max')
    )
    .sort_values('cluster_id')
    .reset_index(drop=True)
)
print(cluster_summary_df.to_string(index=False))

plt.figure(figsize=(8, 5))
results['cluster_id'].value_counts().sort_index().plot(kind='bar', edgecolor='black')
plt.title('Cluster Distribution')
plt.xlabel('Cluster ID')
plt.ylabel('Number of CVEs')
plt.tight_layout()
plt.show()


 cluster_id  cve_count  average_cvss  minimum_cvss  maximum_cvss
          0         12          7.35           7.2           8.4
          1          5          4.26           2.1           5.0
          2          3         10.00          10.0          10.0


C:\Users\Riddhima\AppData\Local\Temp\ipykernel_15008\4207506983.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Vulnerability prioritization score

This is not a supervised prediction model. It is a transparent prioritization score based on documented vulnerability attributes such as CVSS, attack accessibility, privilege requirements, and user interaction requirements. It is intentionally simple and easy to review in code.


In [8]:
def compute_priority_score(df_input: pd.DataFrame) -> pd.Series:
    """Create a transparent prioritization score without claiming supervised prediction."""
    cvss_value = df_input['cvss_score'].fillna(df_input['cvss_score'].median()) / 10.0

    attack_vector_map = {
        'NETWORK': 1.0,
        'ADJACENT': 0.75,
        'LOCAL': 0.40,
        'PHYSICAL': 0.20,
        np.nan: 0.0,
    }
    privilege_map = {
        'NONE': 0.0,
        'LOW': 0.40,
        'HIGH': 0.80,
        'REQUIRED': 0.40,
        np.nan: 0.0,
    }
    user_interaction_map = {
        'NONE': 0.0,
        'REQUIRED': 0.60,
        np.nan: 0.0,
    }
    scope_map = {
        'UNCHANGED': 0.0,
        'CHANGED': 0.35,
        np.nan: 0.0,
    }

    attack_score = df_input['attack_vector'].map(attack_vector_map).fillna(0.0)
    privilege_score = df_input['privileges_required'].map(privilege_map).fillna(0.0)
    user_interaction_score = df_input['user_interaction'].map(user_interaction_map).fillna(0.0)
    scope_score = df_input['scope'].map(scope_map).fillna(0.0)

    score = (
        0.55 * cvss_value
        + 0.25 * attack_score
        + 0.10 * privilege_score
        + 0.10 * user_interaction_score
        + 0.05 * scope_score
    )
    return score.clip(lower=0.0, upper=1.0)

results['ml_priority_score'] = compute_priority_score(results).values
print(results[['cve_id', 'cvss_score', 'cluster_id', 'ml_priority_score']].head().to_string(index=False))
print(f"Priority score range: [{results['ml_priority_score'].min():.3f}, {results['ml_priority_score'].max():.3f}]")


       cve_id  cvss_score  cluster_id  ml_priority_score
CVE-1999-1467        10.0           2             0.8000
CVE-1999-1554         2.1           1             0.2155
CVE-1999-1391         7.2           0             0.4960
CVE-1999-1197         7.2           0             0.4960
CVE-1999-1258         5.0           1             0.5250
Priority score range: [0.216, 0.800]


## 11. Final ML dataset

Create the final CVE-level output with the original characteristics and added ML results.


In [9]:
final_output = results[[
    'cve_id',
    'cvss_score',
    'severity',
    'cwe_id',
    'cwe_name',
    'attack_vector',
    'attack_complexity',
    'privileges_required',
    'user_interaction',
    'scope',
    'cluster_id',
    'ml_priority_score',
]].copy()
display(final_output.head(10))


,cve_id,cvss_score,severity,cwe_id,cwe_name,attack_vector,attack_complexity,privileges_required,user_interaction,scope,cluster_id,ml_priority_score
0,CVE-1999-1467,10.0,HIGH,NVD-CWE-Other,NaN,NETWORK,LOW,NaN,NaN,NaN,2,0.8000
1,CVE-1999-1554,2.1,LOW,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,1,0.2155
2,CVE-1999-1391,7.2,HIGH,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,0,0.4960
3,CVE-1999-1197,7.2,HIGH,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,0,0.4960
4,CVE-1999-1258,5.0,MEDIUM,NVD-CWE-Other,NaN,NETWORK,LOW,NaN,NaN,NaN,1,0.5250
5,CVE-1999-1471,7.2,HIGH,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,0,0.4960
6,CVE-1999-0084,8.4,HIGH,CWE-269,Improper Privilege Management,LOCAL,LOW,NONE,NONE,UNCHANGED,0,0.5620
7,CVE-1999-1198,7.2,HIGH,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,0,0.4960
8,CVE-1999-1438,7.2,HIGH,NVD-CWE-Other,NaN,LOCAL,LOW,NaN,NaN,NaN,0,0.4960
9,CVE-2000-0388,7.5,HIGH,NVD-CWE-Other,NaN,NETWORK,LOW,NaN,NaN,NaN,0,0.6625


## 12. Save results

Persist the outputs to the project data folder.


In [10]:
output_dir = project_root / 'data' / 'features'
output_dir.mkdir(parents=True, exist_ok=True)
cve_output_path = output_dir / 'cve_ml_features.parquet'
summary_output_path = output_dir / 'cluster_summary.parquet'
final_output.to_parquet(cve_output_path, index=False)
cluster_summary_df.to_parquet(summary_output_path, index=False)
print(f"Wrote: {cve_output_path}")
print(f"Wrote: {summary_output_path}")


Wrote: C:\Users\Riddhima\OneDrive\Desktop\cybersecurity-threat-intel\data\features\cve_ml_features.parquet
Wrote: C:\Users\Riddhima\OneDrive\Desktop\cybersecurity-threat-intel\data\features\cluster_summary.parquet


## 13. Reload and validate

Reload the saved parquet files and validate that the output lines up with the development dataset.


In [11]:
reloaded_output = pd.read_parquet(cve_output_path)
reloaded_summary = pd.read_parquet(summary_output_path)
input_count = len(df)
output_count = len(reloaded_output)
duplicate_cve_count = int(reloaded_output['cve_id'].duplicated().sum())
cluster_missing = int(reloaded_output['cluster_id'].isna().sum())
priority_min = float(reloaded_output['ml_priority_score'].min())
priority_max = float(reloaded_output['ml_priority_score'].max())
priority_valid = bool(np.isfinite(reloaded_output['ml_priority_score']).all() and ((reloaded_output['ml_priority_score'] >= 0) & (reloaded_output['ml_priority_score'] <= 1)).all())
summary_count = len(reloaded_summary)
print('Validation report')
print(f"Input CVEs: {input_count}")
print(f"Output CVEs: {output_count}")
print(f"Unique CVEs: {reloaded_output['cve_id'].nunique()}")
print(f"Duplicate CVEs: {duplicate_cve_count}")
print(f"Missing cluster assignments: {cluster_missing}")
print(f"Priority score range: [{priority_min:.3f}, {priority_max:.3f}]")
print(f"Priority score valid: {priority_valid}")
print(f"Cluster summary rows: {summary_count}")
print(f"Cluster counts: {reloaded_output['cluster_id'].value_counts().sort_index().to_dict()}")
print('Reload successful for both Parquet outputs.')


Validation report
Input CVEs: 20
Output CVEs: 20
Unique CVEs: 20
Duplicate CVEs: 0
Missing cluster assignments: 0
Priority score range: [0.216, 0.800]
Priority score valid: True
Cluster summary rows: 3
Cluster counts: {0: 12, 1: 5, 2: 3}
Reload successful for both Parquet outputs.


## 14. Limitations and next steps

- This project currently uses only 20 CVEs, so the clustering result is a development/test demonstration rather than a production-meaningful analysis.
- Cluster IDs are arbitrary and should not be interpreted as safe or dangerous labels.
- The `ml_priority_score` is a transparent prioritization heuristic, not a validated cybersecurity risk model.
- More data, broader NVD ingestion, and cluster stability analysis would be needed before drawing stronger conclusions.
- This notebook intentionally does not implement those future improvements.
